In [367]:
import pandas as pd
import numpy as np 
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score
from xgboost import XGBClassifier
from sklearn.ensemble import RandomForestClassifier

In [368]:
df_train = pd.read_csv('titanic/train.csv')
df_test = pd.read_csv('titanic/test.csv')
if 'Survived' not in df_test.columns:
    df_test['Survived'] = 0

In [369]:
df_train.head()

,PassengerId,Survived,Pclass,Name,Sex,Age,SibSp,Parch,Ticket,Fare,Cabin,Embarked
0,1,0,3,"Braund, Mr. Owen Harris",male,22.0,1,0,A/5 21171,7.2500,NaN,S
1,2,1,1,"Cumings, Mrs. John Bradley (Florence Briggs Th...",female,38.0,1,0,PC 17599,71.2833,C85,C
2,3,1,3,"Heikkinen, Miss. Laina",female,26.0,0,0,STON/O2. 3101282,7.9250,NaN,S
3,4,1,1,"Futrelle, Mrs. Jacques Heath (Lily May Peel)",female,35.0,1,0,113803,53.1000,C123,S
4,5,0,3,"Allen, Mr. William Henry",male,35.0,0,0,373450,8.0500,NaN,S


In [370]:
def Preprocess(df_train, df_test):
    df = pd.concat([df_train, df_test], axis=0)
    
    df = df.drop(['Name', 'Ticket'], axis=1) 
    df['Age'] = df['Age'].fillna(df['Age'].mean())
    df['Cabin'] = df['Cabin'].fillna('X000')
    df['Embarked'] = df['Embarked'].fillna('X')
    df['Fare'] = df['Fare'].fillna(df['Fare'].mean())

    df['cabin_letter'] = df['Cabin'].str.extract(r'([a-zA-Z]+)', expand=False)
    df['Cabin_number'] = df['Cabin'].str.extract(r'(\d+)', expand=False)
    df = df.drop(['Cabin'], axis=1)

    df = pd.get_dummies(df, columns = ['cabin_letter'], prefix = ['Cabin'])
    df = pd.get_dummies(df, columns = ['Embarked'], prefix = ['Embarked'])
    df = pd.get_dummies(df, columns = ['Sex'], prefix = ['Sex'])

    df = df.drop(['Cabin_X'], axis=1)
    df = df.drop(['Embarked_X'], axis=1)

    df['Cabin_number'] = df['Cabin_number'].fillna(0)
    df['Cabin_number'] = pd.to_numeric(df['Cabin_number'])

    df['Pclass_bin_Fare'] = df['Fare'] // df['Pclass']
    df['Pclass_bin_Sex'] = df['Pclass'] - df['Sex_female'] 

    df_train =  df[:len(df_train)]
    df_test =  df[len(df_train):]

    df_test = df_test.drop(['Survived'], axis=1)

    return df_train, df_test


In [371]:
df = pd.concat([df_train, df_test], axis=0)
df.isna().sum()

PassengerId       0
Survived          0
Pclass            0
Name              0
Sex               0
Age             263
SibSp             0
Parch             0
Ticket            0
Fare              1
Cabin          1014
Embarked          2
dtype: int64

In [372]:
train_df, test_df = Preprocess(df_train, df_test)

train_df.corr()['Survived'].sort_values(ascending=False)

Survived           1.000000
Sex_female         0.543351
Pclass_bin_Fare    0.267823
Fare               0.257307
Cabin_number       0.229756
Cabin_B            0.175095
Embarked_C         0.168240
Cabin_D            0.150716
Cabin_E            0.145321
Cabin_C            0.114652
Parch              0.081629
Cabin_F            0.057935
Cabin_A            0.022287
Cabin_G            0.016040
Embarked_Q         0.003650
PassengerId       -0.005007
Cabin_T           -0.026456
SibSp             -0.035322
Age               -0.070323
Embarked_S        -0.155660
Pclass            -0.338481
Pclass_bin_Sex    -0.533994
Sex_male          -0.543351
Name: Survived, dtype: float64

In [373]:
X = train_df.drop(['Survived'], axis=1)
y = train_df['Survived']    

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2)
y_train = np.reshape(y_train, (-1, 1))

In [374]:
X_train.shape, y_train.shape

((712, 22), (712, 1))

In [375]:
model_1 = LogisticRegression()
model_1.fit(X_train, y_train)


/Users/tanvir/Library/Python/3.9/lib/python/site-packages/sklearn/utils/validation.py:1408: DataConversionWarning: A column-vector y was passed when a 1d array was expected. Please change the shape of y to (n_samples, ), for example using ravel().
  y = column_or_1d(y, warn=True)
/Users/tanvir/Library/Python/3.9/lib/python/site-packages/sklearn/linear_model/_linear_loss.py:200: RuntimeWarning: divide by zero encountered in matmul
  raw_prediction = X @ weights + intercept
/Users/tanvir/Library/Python/3.9/lib/python/site-packages/sklearn/linear_model/_linear_loss.py:200: RuntimeWarning: overflow encountered in matmul
  raw_prediction = X @ weights + intercept
/Users/tanvir/Library/Python/3.9/lib/python/site-packages/sklearn/linear_model/_linear_loss.py:200: RuntimeWarning: invalid value encountered in matmul
  raw_prediction = X @ weights + intercept
/Users/tanvir/Library/Python/3.9/lib/python/site-packages/sklearn/linear_model/_linear_loss.py:330: RuntimeWarning: divide by zero encount

LogisticRegression()

In [376]:

y_pred = model_1.predict(X_test)
accuracy_score(y_test, y_pred)

/Users/tanvir/Library/Python/3.9/lib/python/site-packages/sklearn/utils/extmath.py:203: RuntimeWarning: divide by zero encountered in matmul
  ret = a @ b
/Users/tanvir/Library/Python/3.9/lib/python/site-packages/sklearn/utils/extmath.py:203: RuntimeWarning: overflow encountered in matmul
  ret = a @ b
/Users/tanvir/Library/Python/3.9/lib/python/site-packages/sklearn/utils/extmath.py:203: RuntimeWarning: invalid value encountered in matmul
  ret = a @ b


0.8212290502793296

In [377]:
model_2 = XGBClassifier(enable_categorical=True)
model_2.fit(X_train, y_train)

y_pred = model_2.predict(X_test)
accuracy_score(y_test, y_pred)

0.8435754189944135

In [378]:
model_3 = RandomForestClassifier()
model_3.fit(X_train, y_train)

y_pred = model_3.predict(X_test)
accuracy_score(y_test, y_pred)

/Users/tanvir/Library/Python/3.9/lib/python/site-packages/sklearn/base.py:1389: DataConversionWarning: A column-vector y was passed when a 1d array was expected. Please change the shape of y to (n_samples,), for example using ravel().
  return fit_method(estimator, *args, **kwargs)


0.8324022346368715

In [379]:
pred = model_3.predict(test_df)

final = pd.DataFrame()
final['PassengerId'] = test_df['PassengerId']
final['Survived'] = pred

final.to_csv('submission.csv', index=False)